# CADA - Model Benchmarking & Evaluation

This notebook provides a comparative evaluation between **Supervised Classification**, **Unsupervised Anomaly Detection**, and the **CADA Composite Risk Architecture**.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

# Add project root to sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation.benchmark import run_full_benchmark
from src.scoring.cada_scorer import CADACompositeScorer
from src.data.loader import load_motion_data
from src.data.preprocessor import MotionDataPreprocessor
from src.features.kinematics import KinematicFeatureExtractor

## 1. Execute Multi-Model Benchmark Suite

In [ ]:
benchmark_results = run_full_benchmark()
df_bench = pd.DataFrame.from_dict(benchmark_results, orient='index')
display(df_bench[['precision', 'recall', 'f1_score', 'accuracy', 'roc_auc', 'pr_auc', 'spearman_correlation', 'latency_ms_per_sample']])

## 2. Load Scored Test Telemetry & Visualizations

In [ ]:
# Load test data
df_test = load_motion_data('../data/raw/test_motion_data.csv', require_target=True)
preproc = MotionDataPreprocessor()
kinematics = KinematicFeatureExtractor()
df_test_feat = kinematics.fit_transform(preproc.fit_transform(df_test))

# Load trained CADA scorer
scorer = CADACompositeScorer.load('../models/cada_model_bundle.joblib')
df_scored = scorer.score_batch(df_test_feat)
df_scored.head()

## 3. CADA Risk Score Distribution across Driving Classes

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(data=df_scored, x='Class', y='CADA_Score', palette='Set2', inner='quartile')
plt.title('CADA Risk Score Distribution by Ground Truth Driving Class', fontsize=14)
plt.ylabel('CADA Composite Score (0 - 100)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. Risk Component Attribution (Iso vs Stat vs Temporal)

In [ ]:
comp_means = df_scored.groupby('Class')[['Iso_Risk', 'Stat_Risk', 'Temporal_Risk', 'CADA_Score']].mean()
comp_means.plot(kind='bar', figsize=(10, 6))
plt.title('Sub-Component Risk Attribution by Driving Class', fontsize=14)
plt.ylabel('Mean Score (0 - 100)')
plt.xticks(rotation=0)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()